This notebook relies on Papermill to run other notebooks multiple times with different hyperparameters (defined in their papermeter cell).

In [ ]:
import logging
import os
import sys
import papermill as pm

notebook_dir = os.getcwd()
project_dir = os.path.dirname(os.path.dirname(notebook_dir))
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

print(f'Notebook dir: {notebook_dir}\nProject dir: {project_dir}')

logging.getLogger().setLevel(logging.INFO)
%load_ext autoreload
%autoreload 2

In [ ]:
NOTEBOOK_DIR = os.path.join(notebook_dir, "generated_notebooks")
if not os.path.exists(NOTEBOOK_DIR):
    os.makedirs(NOTEBOOK_DIR)

# Sweep on baseline model with motor units spike counts

In [ ]:
# Inititalize a parameters dictionary
parameters = dict(
    subj = "S1",
    sign_mvc = 1,
    sweep_win_size = False,      # Flag to sweep over the window size. If True place results in a subfolder
    load_regression_data_from_file = False,     # assumes there is a pre-generated processed force and mu dataframes.
    profile = False,  # if True, profile the model during prediction time
)
for subject in ["S1", "S2"]:  
    for sign_force in [-1, 1]:
            print(f"{subject} | {sign_force}")
            parameters['sign_mvc'] = sign_force
            parameters['subj'] = subject
            output_notebook_name = f"BaselineRegressionMUCount_{parameters['subj']}_sign_mvc_{parameters['sign_mvc']}.ipynb"
            pm.execute_notebook("BaselineRegressionMUCount.ipynb",
                                os.path.join(NOTEBOOK_DIR, output_notebook_name),
                                parameters=parameters)

# Sweep on baseline model with noisy motor units

In [ ]:
# Inititalize a parameters dictionary
parameters = dict(
    subject = "S1",
    exp_dir = 'ext',
    load_preprocessed_data = True,     # assumes there is a pre-generated emg_dataset object. This is the case if a SNN script has been run before, else set to False
    percent_omission = 0,      # noise percentage
    percent_addition = 0,      # noise percentage
    percent_misattribution = 0,  # noise percentage
    noise_mode = 'misattribution',   # 'omission' | 'addition' | 'misattribution'
)

# omission_noise_levels = [0, 10, 20, 30, 40, 50]
# addition_noise_levels = [1, 2, 3, 4, 5, 10, 15, 20, 25, 30]
misattribution_noise_levels = [50] #[0,1,3,5,10]

for subject in ["S1","S2"]:  
    for exp_dir in ['flex', 'ext']:
        for noise_percentage in misattribution_noise_levels:
            print(f"{subject} | {exp_dir} | noise mode {parameters['noise_mode']} | noise percentage {noise_percentage}")
            parameters['exp_dir'] = exp_dir
            parameters['subject'] = subject
            if parameters['noise_mode'] == 'addition':
                parameters['percent_addition'] = noise_percentage
            elif parameters['noise_mode'] == 'omission':
                parameters['percent_omission'] = noise_percentage
            elif parameters['noise_mode'] == 'misattribution':
                parameters['percent_misattribution'] = noise_percentage
            output_notebook_name = f"BaselineregressionNoise_{parameters['subject']}_exp_dir_{parameters['exp_dir']}_noise_{parameters['noise_mode']}_percent_{noise_percentage}.ipynb"
            pm.execute_notebook("BaselineregressionNoise.ipynb",
                                os.path.join(NOTEBOOK_DIR, output_notebook_name),
                                parameters=parameters)

# Sweep over window size

In [ ]:
windows =[0.01] #+ [round(x * 0.02, 2) for x in range(1, 11)]

parameters = dict(
    subj = "S1",
    sign_mvc = 1,
    win_size_in_sec = 0.08 , # analysis window size in seconds
    sweep_win_size = True,  # Flag to sweep the window size. If True place results in a subfolder
    load_regression_data_from_file = True,  
)
for subject in ["S1"]:  
    for sign_force in [-1]:
        for win_size_in_sec in windows:
            print(f"{subject} | {sign_force} | {win_size_in_sec}")
            parameters['subj'] = subject
            parameters['sign_mvc'] = sign_force
            parameters['win_size_in_sec'] = win_size_in_sec
            output_notebook_name = f"BaselineRegressionMUCount_{parameters['subj']}_sign_mvc_{parameters['sign_mvc']}_win_{win_size_in_sec}.ipynb"
            pm.execute_notebook("BaselineRegressionMUCount.ipynb",
                                os.path.join(NOTEBOOK_DIR, output_notebook_name),
                                parameters=parameters)